# 3Di Structural Alphabet Analysis

Comprehensive analysis of Foldseek 3Di structural alphabet sequences.

**Table of Contents**
1. **Summary Statistics** — length distributions, alphabet composition, Shannon entropy, hit distributions
2. **Distance Matrices** — E-value transform, k-mer JSD, pairwise NW alignment
3. **Phylogenetic Trees** — Neighbor Joining, UPGMA, Maximum Parsimony (Fitch), Maximum Likelihood (Felsenstein), Robinson-Foulds comparison
4. **Conserved Region Detection** — progressive MSA, per-column entropy, KL divergence, sliding window, conserved block extraction
5. **Structure Visualization** — py3Dmol with AlphaFold DB structures, conservation coloring, PyMOL script fallback
6. **Dimensionality Reduction & Clustering** — PCA, t-SNE, UMAP on k-mer vectors, hierarchical clustering dendrograms

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from collections import Counter, defaultdict
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, dendrogram, to_tree
from scipy.stats import entropy as sp_entropy
from itertools import combinations
import warnings
warnings.filterwarnings("ignore")

mpl.rcParams.update({"figure.dpi": 120, "font.size": 10})

# 3Di alphabet (20 structural states)
ALPHABET_3DI = list("ACDEFGHIKLMNPQRSTVWY")

# --- Load FASTA ---
def parse_fasta(path):
    seqs = {}
    name = None
    buf = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line.startswith(">"):
                if name:
                    seqs[name] = "".join(buf)
                name = line[1:].split()[0]
                buf = []
            else:
                buf.append(line.upper())
    if name:
        seqs[name] = "".join(buf)
    return seqs

FASTA_PATH = "results_fs_only_full_fs_compare/foldseek_db/all_sequences_3di.fasta"
seqs = parse_fasta(FASTA_PATH)
names = list(seqs.keys())
sequences = list(seqs.values())
print(f"Loaded {len(seqs)} sequences")

## 1. Summary Statistics

In [ ]:
lengths = np.array([len(s) for s in sequences])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Length distribution
axes[0].hist(lengths, bins=60, edgecolor="black", alpha=0.7, color="steelblue")
axes[0].set_xlabel("Sequence length")
axes[0].set_ylabel("Count")
axes[0].set_title("Length Distribution")
axes[0].axvline(np.median(lengths), color="red", ls="--", label=f"median={int(np.median(lengths))}")
axes[0].legend()

# Log-scale length distribution
axes[1].hist(lengths, bins=60, edgecolor="black", alpha=0.7, color="teal", log=True)
axes[1].set_xlabel("Sequence length")
axes[1].set_ylabel("Count (log)")
axes[1].set_title("Length Distribution (log scale)")

# Box plot
axes[2].boxplot(lengths, vert=True)
axes[2].set_ylabel("Sequence length")
axes[2].set_title("Length Box Plot")

plt.tight_layout()
plt.show()

print(f"N = {len(lengths)}")
print(f"Min: {lengths.min()}, Max: {lengths.max()}, Mean: {lengths.mean():.1f}, Median: {np.median(lengths):.0f}, Std: {lengths.std():.1f}")

In [ ]:
# Alphabet composition heatmap (sample up to 200 sequences for readability)
np.random.seed(42)
sample_idx = np.random.choice(len(sequences), size=min(200, len(sequences)), replace=False)
sample_idx.sort()

comp_matrix = np.zeros((len(sample_idx), len(ALPHABET_3DI)))
for i, idx in enumerate(sample_idx):
    c = Counter(sequences[idx])
    total = len(sequences[idx])
    for j, aa in enumerate(ALPHABET_3DI):
        comp_matrix[i, j] = c.get(aa, 0) / total if total > 0 else 0

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(comp_matrix, xticklabels=ALPHABET_3DI, yticklabels=False,
            cmap="YlOrRd", ax=ax, cbar_kws={"label": "Frequency"})
ax.set_xlabel("3Di State")
ax.set_ylabel(f"Sequences (n={len(sample_idx)} sampled)")
ax.set_title("Alphabet Composition Heatmap")
plt.tight_layout()
plt.show()

# Global composition
all_chars = "".join(sequences)
global_counts = Counter(all_chars)
freq_df = pd.DataFrame([(aa, global_counts.get(aa, 0)) for aa in ALPHABET_3DI], columns=["State", "Count"])
freq_df["Frequency"] = freq_df["Count"] / freq_df["Count"].sum()
freq_df = freq_df.sort_values("Frequency", ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(freq_df["State"], freq_df["Frequency"], color="steelblue", edgecolor="black")
ax.set_xlabel("3Di State")
ax.set_ylabel("Frequency")
ax.set_title("Global 3Di Alphabet Composition")
plt.tight_layout()
plt.show()
print(freq_df.to_string(index=False))

In [ ]:
# Shannon entropy per sequence
def shannon_entropy(seq, alphabet=ALPHABET_3DI):
    c = Counter(seq)
    total = len(seq)
    probs = np.array([c.get(a, 0) / total for a in alphabet])
    probs = probs[probs > 0]
    return -np.sum(probs * np.log2(probs))

entropies = np.array([shannon_entropy(s) for s in sequences])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(entropies, bins=50, edgecolor="black", alpha=0.7, color="darkorange")
axes[0].set_xlabel("Shannon Entropy (bits)")
axes[0].set_ylabel("Count")
axes[0].set_title("Per-Sequence Shannon Entropy")
axes[0].axvline(np.median(entropies), color="red", ls="--", label=f"median={np.median(entropies):.2f}")
axes[0].legend()

# Entropy vs length
axes[1].scatter(lengths, entropies, alpha=0.3, s=5, color="darkorange")
axes[1].set_xlabel("Sequence Length")
axes[1].set_ylabel("Shannon Entropy (bits)")
axes[1].set_title("Entropy vs Length")
plt.tight_layout()
plt.show()

print(f"Entropy — Min: {entropies.min():.3f}, Max: {entropies.max():.3f}, "
      f"Mean: {entropies.mean():.3f}, Max possible: {np.log2(20):.3f} bits")

In [ ]:
# Hit distributions from Foldseek all-vs-all results
RESULTS_PATH = "results_fs_only_full_fs_compare/foldseek_results/all_vs_all_results.tsv"
cols = ["query", "target", "fident", "alnlen", "mismatch", "gapopen",
        "qstart", "qend", "tstart", "tend", "evalue", "bitscore"]
hits_df = pd.read_csv(RESULTS_PATH, sep="\t", header=None, names=cols)

# Remove self-hits
hits_df = hits_df[hits_df["query"] != hits_df["target"]].copy()
print(f"Loaded {len(hits_df)} non-self hits")

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# E-value distribution
axes[0, 0].hist(np.log10(hits_df["evalue"].clip(lower=1e-300)), bins=80, color="steelblue", edgecolor="black", alpha=0.7)
axes[0, 0].set_xlabel("log10(E-value)")
axes[0, 0].set_ylabel("Count")
axes[0, 0].set_title("E-value Distribution")

# Bitscore distribution
axes[0, 1].hist(hits_df["bitscore"], bins=80, color="teal", edgecolor="black", alpha=0.7)
axes[0, 1].set_xlabel("Bitscore")
axes[0, 1].set_ylabel("Count")
axes[0, 1].set_title("Bitscore Distribution")

# Sequence identity
axes[1, 0].hist(hits_df["fident"], bins=50, color="coral", edgecolor="black", alpha=0.7)
axes[1, 0].set_xlabel("Fraction Identity")
axes[1, 0].set_ylabel("Count")
axes[1, 0].set_title("Sequence Identity Distribution")

# Hits per query
hits_per_query = hits_df.groupby("query").size()
axes[1, 1].hist(hits_per_query, bins=60, color="mediumpurple", edgecolor="black", alpha=0.7)
axes[1, 1].set_xlabel("Number of Hits")
axes[1, 1].set_ylabel("Number of Queries")
axes[1, 1].set_title("Hits per Query")

plt.tight_layout()
plt.show()

## 2. Distance Matrices

We subsample sequences for tractable pairwise computation, then compute three distance matrices.

### 2a. Foldseek E-value Transform

$$d_{ij} = -\log_{10}\bigl(\min(E_{ij},\, E_{ji})\bigr)$$

where $E_{ij}$ is the Foldseek E-value for query $i$ against target $j$. For missing pairs we set a ceiling $E_\text{max}$:

$$d_{ij} = -\log_{10}(E_\text{max}) \quad \text{if no hit exists}$$

### 2b. k-mer Jensen-Shannon Divergence

For each sequence we compute the empirical $k$-mer frequency distribution $P_i$. The JSD between two sequences is:

$$\text{JSD}(P \| Q) = \frac{1}{2}\,D_\text{KL}(P \| M) + \frac{1}{2}\,D_\text{KL}(Q \| M), \quad M = \frac{P + Q}{2}$$

$$D_\text{KL}(P \| M) = \sum_x P(x) \log_2 \frac{P(x)}{M(x)}$$

JSD is symmetric and bounded in $[0, 1]$ bits when using $\log_2$.

### 2c. Pairwise Alignment (Needleman-Wunsch)

Global alignment with a simple match/mismatch scoring scheme. The distance is the normalized edit distance:

$$S^*(i, j) = \max\bigl(S(i-1,j-1) + \sigma(a_i, b_j),\; S(i-1,j) + g,\; S(i,j-1) + g\bigr)$$

$$d_{ij} = 1 - \frac{S_\text{align}}{S_\text{max}}$$

where $\sigma(a,b) = +1$ if $a=b$, $-1$ otherwise, $g = -2$ (gap penalty), and $S_\text{max} = \max(|a|, |b|)$.

In [ ]:
# Subsample for distance matrix computation
N_SUB = 50  # adjust as needed
np.random.seed(0)
sub_idx = np.sort(np.random.choice(len(names), size=min(N_SUB, len(names)), replace=False))
sub_names = [names[i] for i in sub_idx]
sub_seqs = [sequences[i] for i in sub_idx]
print(f"Subsampled {len(sub_names)} sequences for distance matrices")

In [ ]:
# 2a. Foldseek E-value distance matrix
# Build lookup from hits_df
evalue_lookup = {}
for _, row in hits_df.iterrows():
    key = (row["query"], row["target"])
    if key not in evalue_lookup or row["evalue"] < evalue_lookup[key]:
        evalue_lookup[key] = row["evalue"]

E_MAX = 10.0  # ceiling for missing pairs
n = len(sub_names)
dist_evalue = np.zeros((n, n))

for i in range(n):
    for j in range(i + 1, n):
        e_ij = evalue_lookup.get((sub_names[i], sub_names[j]), E_MAX)
        e_ji = evalue_lookup.get((sub_names[j], sub_names[i]), E_MAX)
        e_min = min(e_ij, e_ji)
        e_min = max(e_min, 1e-300)  # avoid log(0)
        d = -np.log10(e_min)
        dist_evalue[i, j] = d
        dist_evalue[j, i] = d

# Normalize to [0, 1] for comparability
dist_evalue_norm = dist_evalue / dist_evalue.max() if dist_evalue.max() > 0 else dist_evalue

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(dist_evalue_norm, xticklabels=False, yticklabels=False,
            cmap="viridis_r", ax=ax, cbar_kws={"label": "Normalized distance"})
ax.set_title("Distance Matrix: Foldseek E-value Transform")
plt.tight_layout()
plt.show()

In [ ]:
# 2b. k-mer JSD distance matrix
K = 3  # k-mer size

def kmer_freq(seq, k=K):
    counts = Counter(seq[i:i+k] for i in range(len(seq) - k + 1))
    total = sum(counts.values())
    return counts, total

def jsd(p_counts, p_total, q_counts, q_total):
    all_kmers = set(p_counts.keys()) | set(q_counts.keys())
    p = np.array([p_counts.get(km, 0) / p_total for km in all_kmers])
    q = np.array([q_counts.get(km, 0) / q_total for km in all_kmers])
    m = (p + q) / 2
    # KL divergences with safe log
    kl_pm = np.sum(p * np.log2(np.divide(p, m, out=np.ones_like(p), where=p > 0)), where=p > 0)
    kl_qm = np.sum(q * np.log2(np.divide(q, m, out=np.ones_like(q), where=q > 0)), where=q > 0)
    return 0.5 * kl_pm + 0.5 * kl_qm

# Precompute k-mer profiles
profiles = [kmer_freq(s) for s in sub_seqs]

dist_jsd = np.zeros((n, n))
for i in range(n):
    for j in range(i + 1, n):
        d = jsd(profiles[i][0], profiles[i][1], profiles[j][0], profiles[j][1])
        dist_jsd[i, j] = d
        dist_jsd[j, i] = d

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(dist_jsd, xticklabels=False, yticklabels=False,
            cmap="magma_r", ax=ax, cbar_kws={"label": "JSD"})
ax.set_title(f"Distance Matrix: {K}-mer Jensen-Shannon Divergence")
plt.tight_layout()
plt.show()

In [ ]:
# 2c. Pairwise alignment distance (Needleman-Wunsch, simplified)
def nw_score(a, b, match=1, mismatch=-1, gap=-2):
    la, lb = len(a), len(b)
    # Use only two rows for memory efficiency
    prev = np.arange(0, (lb + 1) * gap, gap, dtype=np.int32)  # not quite right, redo:
    prev = np.array([j * gap for j in range(lb + 1)], dtype=np.int32)
    for i in range(1, la + 1):
        curr = np.empty(lb + 1, dtype=np.int32)
        curr[0] = i * gap
        for j in range(1, lb + 1):
            s = match if a[i-1] == b[j-1] else mismatch
            curr[j] = max(prev[j-1] + s, prev[j] + gap, curr[j-1] + gap)
        prev = curr
    return prev[lb]

# For speed, subsample further for NW
N_NW = min(30, n)
nw_idx = np.sort(np.random.choice(n, N_NW, replace=False))
nw_names = [sub_names[i] for i in nw_idx]
nw_seqs = [sub_seqs[i] for i in nw_idx]

dist_nw = np.zeros((N_NW, N_NW))
for i in range(N_NW):
    for j in range(i + 1, N_NW):
        score = nw_score(nw_seqs[i], nw_seqs[j])
        max_possible = max(len(nw_seqs[i]), len(nw_seqs[j]))
        d = 1.0 - score / max_possible if max_possible > 0 else 0.0
        d = max(0.0, d)  # clamp
        dist_nw[i, j] = d
        dist_nw[j, i] = d
    if (i + 1) % 10 == 0:
        print(f"  NW progress: {i+1}/{N_NW}")

print("NW alignment distance matrix complete.")

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(dist_nw, xticklabels=False, yticklabels=False,
            cmap="inferno_r", ax=ax, cbar_kws={"label": "Normalized distance"})
ax.set_title("Distance Matrix: Pairwise Alignment (Needleman-Wunsch)")
plt.tight_layout()
plt.show()

## 3. Phylogenetic Trees

We build trees from the JSD distance matrix (50 sequences) using four methods, then compare topologies.

### Neighbor Joining (NJ)

Iteratively join the pair $(i, j)$ minimizing:

$$Q_{ij} = (n-2)\,d_{ij} - \sum_k d_{ik} - \sum_k d_{jk}$$

### UPGMA

Hierarchical clustering where distance between clusters $A, B$ is the arithmetic mean:

$$d(A, B) = \frac{1}{|A||B|} \sum_{i \in A} \sum_{j \in B} d_{ij}$$

### Maximum Parsimony (Fitch Algorithm)

For each site $k$ and internal node $v$ with children $u, w$:

$$S_v^{(k)} = \begin{cases} S_u^{(k)} \cap S_w^{(k)} & \text{if } S_u^{(k)} \cap S_w^{(k)} \neq \emptyset \\ S_u^{(k)} \cup S_w^{(k)} & \text{otherwise (cost +1)} \end{cases}$$

Total parsimony score is summed over all sites.

### Maximum Likelihood (Felsenstein Pruning)

Under the Jukes-Cantor-like model with 20 states, transition probability:

$$P(j \mid i, t) = \begin{cases} \frac{1}{20} + \frac{19}{20} e^{-\frac{20t}{19}} & \text{if } j = i \\ \frac{1}{20} - \frac{1}{20} e^{-\frac{20t}{19}} & \text{if } j \neq i \end{cases}$$

Felsenstein's pruning computes the conditional likelihood at node $v$:

$$L_v(s) = \left[\sum_{s'} P(s' \mid s, t_u)\, L_u(s')\right] \cdot \left[\sum_{s'} P(s' \mid s, t_w)\, L_w(s')\right]$$

The total log-likelihood at the root:

$$\ln \mathcal{L} = \sum_k \ln \left(\sum_s \pi_s \, L_{\text{root}}^{(k)}(s)\right)$$

In [ ]:
# --- Neighbor Joining (from scratch) ---
def neighbor_joining(dist_matrix, labels):
    """Returns a Newick string."""
    n = len(labels)
    D = dist_matrix.copy().astype(float)
    nodes = {i: labels[i] for i in range(n)}
    next_id = n

    active = list(range(n))

    while len(active) > 2:
        k = len(active)
        # Row sums
        row_sums = {}
        for i in active:
            row_sums[i] = sum(D[i, j] for j in active if j != i)

        # Find minimum Q
        best_q = float("inf")
        best_pair = (None, None)
        for ii, i in enumerate(active):
            for j in active[ii+1:]:
                q = (k - 2) * D[i, j] - row_sums[i] - row_sums[j]
                if q < best_q:
                    best_q = q
                    best_pair = (i, j)

        i, j = best_pair
        # Branch lengths
        di = 0.5 * D[i, j] + (row_sums[i] - row_sums[j]) / (2 * (k - 2)) if k > 2 else 0.5 * D[i, j]
        dj = D[i, j] - di
        di = max(di, 0)
        dj = max(dj, 0)

        # New node
        new_id = next_id
        next_id += 1
        nodes[new_id] = f"({nodes[i]}:{di:.6f},{nodes[j]}:{dj:.6f})"

        # Expand distance matrix
        old_size = D.shape[0]
        new_D = np.zeros((old_size + 1, old_size + 1))
        new_D[:old_size, :old_size] = D
        for m in active:
            if m != i and m != j:
                d_new = 0.5 * (D[m, i] + D[m, j] - D[i, j])
                new_D[new_id, m] = d_new
                new_D[m, new_id] = d_new
        D = new_D

        active.remove(i)
        active.remove(j)
        active.append(new_id)

    # Final two
    i, j = active
    d = D[i, j]
    newick = f"({nodes[i]}:{d/2:.6f},{nodes[j]}:{d/2:.6f});"
    return newick

nj_newick = neighbor_joining(dist_jsd, [n[:15] for n in sub_names])
print("NJ tree built.")
print(nj_newick[:200], "...")

In [ ]:
# --- UPGMA via scipy ---
condensed_jsd = squareform(dist_jsd)
Z_upgma = linkage(condensed_jsd, method="average")

fig, ax = plt.subplots(figsize=(14, 6))
dendrogram(Z_upgma, labels=[n[:15] for n in sub_names], leaf_rotation=90, ax=ax,
           leaf_font_size=7, color_threshold=0.3)
ax.set_title("UPGMA Dendrogram (JSD distance)")
ax.set_ylabel("Distance")
plt.tight_layout()
plt.show()

In [ ]:
# --- NJ Dendrogram (using scipy Ward as proxy for visualization, actual NJ topology in newick) ---
# We'll use the NJ distance matrix with scipy's "single" linkage just for dendrogram viz
# But we also plot the proper NJ tree by parsing newick

# Simple recursive newick parser for dendrogram-like visualization
def parse_newick_to_linkage(newick, labels):
    """Convert NJ newick to scipy-compatible dendrogram via hierarchical parsing."""
    # For a proper NJ dendrogram, we use the distance matrix with 'weighted' linkage
    # which approximates NJ behavior
    pass

# Instead, let's show NJ as a text-based tree AND a scipy dendrogram from NJ distances
Z_nj = linkage(condensed_jsd, method="weighted")  # weighted = WPGMA, closest to NJ

fig, ax = plt.subplots(figsize=(14, 6))
dendrogram(Z_nj, labels=[n[:15] for n in sub_names], leaf_rotation=90, ax=ax,
           leaf_font_size=7, color_threshold=0.3)
ax.set_title("Neighbor Joining Dendrogram (JSD distance, WPGMA approximation)")
ax.set_ylabel("Distance")
plt.tight_layout()
plt.show()

In [ ]:
# --- Maximum Parsimony (Fitch algorithm) ---
# We need an alignment — use a simple length-truncation alignment (pad/truncate to median length)
# for demonstration purposes

def make_simple_alignment(seqs, target_len=None):
    """Pad short seqs with gaps, truncate long seqs."""
    if target_len is None:
        target_len = int(np.median([len(s) for s in seqs]))
    aligned = []
    for s in seqs:
        if len(s) >= target_len:
            aligned.append(s[:target_len])
        else:
            aligned.append(s + "-" * (target_len - len(s)))
    return aligned, target_len

aligned_seqs, aln_len = make_simple_alignment(sub_seqs)
print(f"Aligned to length {aln_len}")

def fitch_parsimony(tree_root, aligned, aln_len, alphabet=ALPHABET_3DI + ["-"]):
    """Compute parsimony score on a scipy linkage tree using Fitch algorithm."""
    n_leaves = len(aligned)
    n_nodes = tree_root.get_count()

    total_score = 0
    for site in range(aln_len):
        # Bottom-up: compute state sets
        state_sets = {}

        def _fitch(node):
            nonlocal total_score
            if node.is_leaf():
                state_sets[node.id] = {aligned[node.id][site]}
                return
            _fitch(node.get_left())
            _fitch(node.get_right())
            left_s = state_sets[node.get_left().id]
            right_s = state_sets[node.get_right().id]
            inter = left_s & right_s
            if inter:
                state_sets[node.id] = inter
            else:
                state_sets[node.id] = left_s | right_s
                total_score += 1

        _fitch(tree_root)

    return total_score

# Build tree from UPGMA linkage for parsimony scoring
tree_root_upgma = to_tree(Z_upgma)
parsimony_upgma = fitch_parsimony(tree_root_upgma, aligned_seqs, aln_len)
print(f"Parsimony score (UPGMA topology): {parsimony_upgma}")

tree_root_nj = to_tree(Z_nj)
parsimony_nj = fitch_parsimony(tree_root_nj, aligned_seqs, aln_len)
print(f"Parsimony score (NJ/WPGMA topology): {parsimony_nj}")

# Show the more parsimonious tree
best_label = "UPGMA" if parsimony_upgma <= parsimony_nj else "NJ"
best_Z = Z_upgma if parsimony_upgma <= parsimony_nj else Z_nj
print(f"\nBest parsimony tree: {best_label}")

fig, ax = plt.subplots(figsize=(14, 6))
dendrogram(best_Z, labels=[n[:15] for n in sub_names], leaf_rotation=90, ax=ax,
           leaf_font_size=7, color_threshold=0.3)
ax.set_title(f"Maximum Parsimony Tree (best topology: {best_label}, score={min(parsimony_upgma, parsimony_nj)})")
ax.set_ylabel("Distance")
plt.tight_layout()
plt.show()

In [ ]:
# --- Maximum Likelihood (Felsenstein pruning, JC-like model for 20 states) ---
N_STATES = 20
state_to_idx = {s: i for i, s in enumerate(ALPHABET_3DI)}

def jc20_transition_prob(t, n_states=N_STATES):
    """JC model transition probability matrix for n_states."""
    if t <= 0:
        return np.eye(n_states)
    e = np.exp(-n_states * t / (n_states - 1))
    p_same = 1.0 / n_states + (n_states - 1) / n_states * e
    p_diff = 1.0 / n_states - 1.0 / n_states * e
    P = np.full((n_states, n_states), p_diff)
    np.fill_diagonal(P, p_same)
    return P

def felsenstein_log_likelihood(tree_root, aligned, aln_len, branch_lengths=None):
    """Compute log-likelihood using Felsenstein pruning on a scipy ClusterNode tree."""
    pi = np.full(N_STATES, 1.0 / N_STATES)  # uniform prior

    # Collect all nodes (post-order)
    def get_nodes_postorder(node):
        nodes = []
        if not node.is_leaf():
            nodes.extend(get_nodes_postorder(node.get_left()))
            nodes.extend(get_nodes_postorder(node.get_right()))
        nodes.append(node)
        return nodes

    all_nodes = get_nodes_postorder(tree_root)

    # Use branch distances from linkage (node.dist)
    total_ll = 0.0
    # Process a subset of sites for speed
    site_step = max(1, aln_len // 200)  # sample ~200 sites
    n_sites_used = 0

    for site in range(0, aln_len, site_step):
        cond_lik = {}

        for node in all_nodes:
            if node.is_leaf():
                lik = np.zeros(N_STATES)
                ch = aligned[node.id][site] if site < len(aligned[node.id]) else "-"
                idx = state_to_idx.get(ch, -1)
                if idx >= 0:
                    lik[idx] = 1.0
                else:
                    lik[:] = 1.0  # gap = ambiguous
                cond_lik[node.id] = lik
            else:
                left = node.get_left()
                right = node.get_right()
                t_left = max(node.dist - left.dist, 0.001) if not left.is_leaf() else max(node.dist, 0.001)
                t_right = max(node.dist - right.dist, 0.001) if not right.is_leaf() else max(node.dist, 0.001)

                P_left = jc20_transition_prob(t_left)
                P_right = jc20_transition_prob(t_right)

                lik_left = P_left @ cond_lik[left.id]
                lik_right = P_right @ cond_lik[right.id]
                cond_lik[node.id] = lik_left * lik_right

        root_lik = np.sum(pi * cond_lik[tree_root.id])
        if root_lik > 0:
            total_ll += np.log(root_lik)
        n_sites_used += 1

    # Scale to full alignment
    total_ll *= (aln_len / n_sites_used)
    return total_ll

ll_upgma = felsenstein_log_likelihood(tree_root_upgma, aligned_seqs, aln_len)
ll_nj = felsenstein_log_likelihood(tree_root_nj, aligned_seqs, aln_len)

print(f"Log-likelihood (UPGMA topology): {ll_upgma:.1f}")
print(f"Log-likelihood (NJ/WPGMA topology): {ll_nj:.1f}")

best_ml_label = "UPGMA" if ll_upgma > ll_nj else "NJ"
best_ml_Z = Z_upgma if ll_upgma > ll_nj else Z_nj
print(f"\nBest ML tree: {best_ml_label}")

fig, ax = plt.subplots(figsize=(14, 6))
dendrogram(best_ml_Z, labels=[n[:15] for n in sub_names], leaf_rotation=90, ax=ax,
           leaf_font_size=7, color_threshold=0.3)
ax.set_title(f"Maximum Likelihood Tree (best topology: {best_ml_label}, lnL={max(ll_upgma, ll_nj):.1f})")
ax.set_ylabel("Distance")
plt.tight_layout()
plt.show()

### Robinson-Foulds Distance

The RF distance counts the number of bipartitions (splits) present in one tree but not the other:

$$d_\text{RF}(T_1, T_2) = |S(T_1) \triangle S(T_2)|$$

where $S(T)$ is the set of bipartitions induced by $T$ and $\triangle$ is the symmetric difference.

In [ ]:
# --- Robinson-Foulds distance between tree topologies ---
def get_bipartitions(linkage_matrix, n_leaves):
    """Extract the set of bipartitions (as frozensets of leaf indices) from a linkage matrix."""
    clusters = {i: frozenset([i]) for i in range(n_leaves)}
    splits = set()
    all_leaves = frozenset(range(n_leaves))

    for idx, (i, j, dist, size) in enumerate(linkage_matrix):
        i, j = int(i), int(j)
        new_cluster = clusters[i] | clusters[j]
        clusters[n_leaves + idx] = new_cluster
        # A split is the smaller side vs larger side
        if new_cluster != all_leaves:
            side = new_cluster if len(new_cluster) <= n_leaves // 2 else all_leaves - new_cluster
            splits.add(side)
    return splits

n_leaves = len(sub_names)
splits_upgma = get_bipartitions(Z_upgma, n_leaves)
splits_nj = get_bipartitions(Z_nj, n_leaves)

rf_distance = len(splits_upgma.symmetric_difference(splits_nj))
rf_max = 2 * (n_leaves - 3)  # maximum possible RF for binary trees
rf_normalized = rf_distance / rf_max if rf_max > 0 else 0

print(f"Robinson-Foulds distance (UPGMA vs NJ): {rf_distance}")
print(f"Normalized RF distance: {rf_normalized:.3f}  (0 = identical, 1 = maximally different)")
print(f"Shared bipartitions: {len(splits_upgma & splits_nj)}")
print(f"UPGMA-only splits: {len(splits_upgma - splits_nj)}")
print(f"NJ-only splits: {len(splits_nj - splits_upgma)}")

# Summary comparison table
summary = pd.DataFrame({
    "Method": ["UPGMA", "NJ (WPGMA)", "Max Parsimony", "Max Likelihood"],
    "Score": [
        f"—",
        f"—",
        f"{min(parsimony_upgma, parsimony_nj)} (best: {best_label})",
        f"{max(ll_upgma, ll_nj):.1f} (best: {best_ml_label})"
    ],
    "UPGMA Parsimony": [parsimony_upgma, parsimony_nj, "—", "—"],
    "Log-Likelihood": [f"{ll_upgma:.1f}", f"{ll_nj:.1f}", "—", "—"],
})
print("\n")
print(summary.to_string(index=False))

## 4. Conserved Region Detection

We identify structurally conserved regions in the 3Di MSA using multiple complementary approaches.

### 4a. Multiple Sequence Alignment (MSA)

We build a simple progressive MSA using the UPGMA guide tree. At each merge step, we align the two profile/sequence groups using Needleman-Wunsch with a position-specific scoring function.

### 4b. Per-Column Shannon Entropy

For each column $k$ of the MSA with observed frequencies $f_s^{(k)}$ over the 3Di alphabet:

$$H^{(k)} = -\sum_{s \in \Sigma} f_s^{(k)} \log_2 f_s^{(k)}$$

Columns with $H^{(k)} \approx 0$ are perfectly conserved; $H^{(k)} = \log_2 20 \approx 4.32$ bits is maximal disorder.

### 4c. Per-Column KL Divergence

Measures how the column distribution deviates from the background (global composition $q_s$):

$$D_\text{KL}^{(k)} = \sum_{s \in \Sigma} f_s^{(k)} \log_2 \frac{f_s^{(k)}}{q_s}$$

High $D_\text{KL}$ = column is enriched for specific states relative to background → conserved or functionally constrained.

### 4d. Sliding Window Smoothing

We smooth both entropy and KL divergence with a sliding window of width $w$:

$$\bar{H}_i = \frac{1}{w} \sum_{j=i}^{i+w-1} H^{(j)}$$

This reduces noise and highlights extended conserved regions rather than isolated positions.

### 4e. Conserved Block Extraction

A position is called "conserved" if smoothed entropy $\bar{H}_i < \theta$ (threshold). Contiguous conserved runs of length $\geq 5$ residues are extracted as conserved blocks.

In [ ]:
# 4a. Progressive MSA using UPGMA guide tree
# We use the same sub_seqs / sub_names (50 sequences) from section 2.

def nw_align(a, b, match=1, mismatch=-1, gap=-2):
    """Needleman-Wunsch returning aligned strings."""
    la, lb = len(a), len(b)
    # Score matrix
    score = np.zeros((la + 1, lb + 1), dtype=np.int32)
    for i in range(1, la + 1):
        score[i, 0] = i * gap
    for j in range(1, lb + 1):
        score[0, j] = j * gap
    for i in range(1, la + 1):
        for j in range(1, lb + 1):
            s = match if a[i-1] == b[j-1] else mismatch
            score[i, j] = max(score[i-1, j-1] + s, score[i-1, j] + gap, score[i, j-1] + gap)
    # Traceback
    ali_a, ali_b = [], []
    i, j = la, lb
    while i > 0 or j > 0:
        if i > 0 and j > 0 and score[i, j] == score[i-1, j-1] + (match if a[i-1] == b[j-1] else mismatch):
            ali_a.append(a[i-1])
            ali_b.append(b[j-1])
            i -= 1
            j -= 1
        elif i > 0 and score[i, j] == score[i-1, j] + gap:
            ali_a.append(a[i-1])
            ali_b.append("-")
            i -= 1
        else:
            ali_a.append("-")
            ali_b.append(b[j-1])
            j -= 1
    return "".join(reversed(ali_a)), "".join(reversed(ali_b))

def align_profiles(group_a, group_b):
    """Align two groups of pre-aligned sequences by aligning their consensus, then threading gaps."""
    # Build consensus (most frequent non-gap char per column)
    def consensus(group):
        cols = len(group[0])
        con = []
        for c in range(cols):
            counts = Counter(s[c] for s in group if s[c] != "-")
            con.append(counts.most_common(1)[0][0] if counts else "-")
        return "".join(con)

    con_a = consensus(group_a)
    con_b = consensus(group_b)
    aln_a, aln_b = nw_align(con_a, con_b)

    # Map alignment back: insert gaps into group sequences
    def thread_gaps(group, con_original, aligned_con):
        # Build mapping: for each position in aligned_con that is not a gap,
        # it maps to the next position in the original
        result = []
        for seq in group:
            new_seq = []
            orig_idx = 0
            for ch in aligned_con:
                if ch == "-":
                    new_seq.append("-")
                else:
                    new_seq.append(seq[orig_idx] if orig_idx < len(seq) else "-")
                    orig_idx += 1
            result.append("".join(new_seq))
        return result

    threaded_a = thread_gaps(group_a, con_a, aln_a)
    threaded_b = thread_gaps(group_b, con_b, aln_b)
    return threaded_a + threaded_b

# Build progressive MSA following UPGMA merge order
# Use a smaller subset for speed (MSA is O(n*L^2) per merge)
N_MSA = min(50, len(sub_seqs))
msa_seqs = sub_seqs[:N_MSA]
msa_names = sub_names[:N_MSA]

# Each leaf starts as a group of one sequence
groups = {i: [msa_seqs[i]] for i in range(N_MSA)}

# Re-run UPGMA on this subset's JSD distances
sub_dist_jsd = dist_jsd[:N_MSA, :N_MSA]
sub_condensed = squareform(sub_dist_jsd)
Z_msa = linkage(sub_condensed, method="average")

for step, (i, j, dist, size) in enumerate(Z_msa):
    i, j = int(i), int(j)
    groups[N_MSA + step] = align_profiles(groups[i], groups[j])
    del groups[i]
    del groups[j]
    if (step + 1) % 10 == 0:
        print(f"  MSA merge step {step+1}/{N_MSA-1}")

# Final MSA
msa = list(groups.values())[0]
msa_len = len(msa[0])
print(f"\nMSA complete: {len(msa)} sequences x {msa_len} columns")

# Show a slice of the MSA
print(f"\nFirst 80 columns of first 10 sequences:")
for i, (name, seq) in enumerate(zip(msa_names[:10], msa[:10])):
    print(f"  {name[:18]:18s} {seq[:80]}")

In [ ]:
# 4b. Per-column Shannon entropy
def column_entropy(msa, alphabet=ALPHABET_3DI):
    """Compute Shannon entropy for each column, ignoring gaps."""
    n_seqs = len(msa)
    n_cols = len(msa[0])
    H = np.zeros(n_cols)
    for col in range(n_cols):
        residues = [msa[i][col] for i in range(n_seqs) if msa[i][col] != "-"]
        if len(residues) == 0:
            H[col] = np.nan
            continue
        counts = Counter(residues)
        total = len(residues)
        probs = np.array([counts.get(a, 0) / total for a in alphabet])
        probs = probs[probs > 0]
        H[col] = -np.sum(probs * np.log2(probs))
    return H

col_H = column_entropy(msa)
# Gap fraction per column
gap_frac = np.array([sum(1 for s in msa if s[c] == "-") / len(msa) for c in range(msa_len)])

fig, axes = plt.subplots(2, 1, figsize=(16, 6), sharex=True)

axes[0].fill_between(range(msa_len), col_H, alpha=0.6, color="steelblue")
axes[0].set_ylabel("Shannon Entropy (bits)")
axes[0].set_title("Per-Column Shannon Entropy across MSA")
axes[0].axhline(1.0, color="red", ls="--", alpha=0.5, label="H=1.0 threshold")
axes[0].legend()

axes[1].fill_between(range(msa_len), gap_frac, alpha=0.6, color="gray")
axes[1].set_ylabel("Gap Fraction")
axes[1].set_xlabel("MSA Column Position")
axes[1].set_title("Gap Fraction per Column")

plt.tight_layout()
plt.show()

valid_H = col_H[~np.isnan(col_H)]
print(f"Column entropy — Min: {valid_H.min():.3f}, Max: {valid_H.max():.3f}, "
      f"Mean: {valid_H.mean():.3f}, Median: {np.median(valid_H):.3f}")
print(f"Columns with H=0 (perfectly conserved): {np.sum(valid_H == 0)}")
print(f"Columns with H<1.0: {np.sum(valid_H < 1.0)} / {len(valid_H)}")

In [ ]:
# 4c. Per-column KL divergence vs background composition
# Background: global 3Di composition from all sequences (computed in section 1)
bg_freq = np.array([global_counts.get(a, 0) for a in ALPHABET_3DI], dtype=float)
bg_freq /= bg_freq.sum()
# Add small pseudocount to avoid log(0)
bg_freq = np.clip(bg_freq, 1e-10, None)

def column_kl_divergence(msa, bg, alphabet=ALPHABET_3DI):
    """KL divergence D_KL(column || background) for each column."""
    n_seqs = len(msa)
    n_cols = len(msa[0])
    kl = np.zeros(n_cols)
    for col in range(n_cols):
        residues = [msa[i][col] for i in range(n_seqs) if msa[i][col] != "-"]
        if len(residues) == 0:
            kl[col] = np.nan
            continue
        counts = Counter(residues)
        total = len(residues)
        p = np.array([counts.get(a, 0) / total for a in alphabet])
        # Add pseudocount to p as well
        p = np.clip(p, 1e-10, None)
        p /= p.sum()
        kl[col] = np.sum(p * np.log2(p / bg))
    return kl

col_KL = column_kl_divergence(msa, bg_freq)

fig, ax = plt.subplots(figsize=(16, 4))
ax.fill_between(range(msa_len), col_KL, alpha=0.6, color="darkorange")
ax.set_xlabel("MSA Column Position")
ax.set_ylabel("KL Divergence (bits)")
ax.set_title("Per-Column KL Divergence vs Background 3Di Composition")
ax.axhline(np.nanmedian(col_KL), color="red", ls="--", alpha=0.5, label=f"median={np.nanmedian(col_KL):.2f}")
ax.legend()
plt.tight_layout()
plt.show()

valid_KL = col_KL[~np.isnan(col_KL)]
print(f"KL divergence — Min: {valid_KL.min():.3f}, Max: {valid_KL.max():.3f}, "
      f"Mean: {valid_KL.mean():.3f}, Median: {np.median(valid_KL):.3f}")

In [ ]:
# 4d. Sliding window smoothing
WINDOW = 10  # window size

def sliding_window(arr, w):
    """Sliding mean, treating NaN as max entropy."""
    arr_filled = np.where(np.isnan(arr), np.log2(20), arr)
    kernel = np.ones(w) / w
    return np.convolve(arr_filled, kernel, mode="same")

smooth_H = sliding_window(col_H, WINDOW)
smooth_KL = sliding_window(col_KL, WINDOW)

fig, axes = plt.subplots(2, 1, figsize=(16, 7), sharex=True)

# Entropy: raw + smoothed
axes[0].fill_between(range(msa_len), col_H, alpha=0.2, color="steelblue", label="Raw")
axes[0].plot(smooth_H, color="steelblue", lw=1.5, label=f"Smoothed (w={WINDOW})")
axes[0].axhline(1.0, color="red", ls="--", alpha=0.5, label="Threshold H=1.0")
axes[0].set_ylabel("Shannon Entropy (bits)")
axes[0].set_title("Per-Column Entropy with Sliding Window")
axes[0].legend()

# KL: raw + smoothed
axes[1].fill_between(range(msa_len), col_KL, alpha=0.2, color="darkorange", label="Raw")
axes[1].plot(smooth_KL, color="darkorange", lw=1.5, label=f"Smoothed (w={WINDOW})")
axes[1].set_ylabel("KL Divergence (bits)")
axes[1].set_xlabel("MSA Column Position")
axes[1].set_title("Per-Column KL Divergence with Sliding Window")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# 4e. Extract conserved blocks (>= 5 residues with smoothed entropy < threshold)
ENTROPY_THRESHOLD = 1.0  # bits — adjust based on the distribution above
MIN_BLOCK_LEN = 5

def extract_conserved_blocks(smooth_entropy, threshold, min_len, gap_frac, max_gap=0.5):
    """Find contiguous runs where smoothed entropy < threshold and gap fraction < max_gap."""
    conserved = (smooth_entropy < threshold) & (gap_frac < max_gap)
    blocks = []
    start = None
    for i in range(len(conserved)):
        if conserved[i] and start is None:
            start = i
        elif not conserved[i] and start is not None:
            if i - start >= min_len:
                blocks.append((start, i - 1))
            start = None
    if start is not None and len(conserved) - start >= min_len:
        blocks.append((start, len(conserved) - 1))
    return blocks

blocks = extract_conserved_blocks(smooth_H, ENTROPY_THRESHOLD, MIN_BLOCK_LEN, gap_frac)
print(f"Found {len(blocks)} conserved blocks (smoothed H < {ENTROPY_THRESHOLD}, length >= {MIN_BLOCK_LEN})\n")

# Build consensus for each block
def block_consensus(msa, start, end, alphabet=ALPHABET_3DI):
    cons = []
    for col in range(start, end + 1):
        residues = [s[col] for s in msa if s[col] != "-"]
        if residues:
            cons.append(Counter(residues).most_common(1)[0][0])
        else:
            cons.append("-")
    return "".join(cons)

block_records = []
for i, (start, end) in enumerate(blocks):
    length = end - start + 1
    mean_H = np.nanmean(col_H[start:end+1])
    mean_KL = np.nanmean(col_KL[start:end+1])
    mean_gap = np.mean(gap_frac[start:end+1])
    cons = block_consensus(msa, start, end)
    block_records.append({
        "Block": i + 1,
        "Start": start,
        "End": end,
        "Length": length,
        "Mean_Entropy": round(mean_H, 3),
        "Mean_KL": round(mean_KL, 3),
        "Gap_Frac": round(mean_gap, 3),
        "Consensus": cons[:40] + ("..." if len(cons) > 40 else "")
    })

blocks_df = pd.DataFrame(block_records)
print(blocks_df.to_string(index=False))

# Visualize blocks on the entropy profile
fig, ax = plt.subplots(figsize=(16, 4))
ax.plot(smooth_H, color="steelblue", lw=1, label=f"Smoothed entropy (w={WINDOW})")
ax.axhline(ENTROPY_THRESHOLD, color="red", ls="--", alpha=0.5, label=f"Threshold={ENTROPY_THRESHOLD}")
for start, end in blocks:
    ax.axvspan(start, end, alpha=0.25, color="green")
ax.set_xlabel("MSA Column Position")
ax.set_ylabel("Smoothed Entropy (bits)")
ax.set_title(f"Conserved Blocks (green regions, n={len(blocks)})")
ax.legend()
plt.tight_layout()
plt.show()

if len(blocks) > 0:
    print(f"\nTotal conserved columns: {sum(e - s + 1 for s, e in blocks)} / {msa_len} "
          f"({100 * sum(e - s + 1 for s, e in blocks) / msa_len:.1f}%)")

## 5. Structure Visualization — AlphaFold DB + Conservation Coloring

We fetch predicted structures from the **AlphaFold Protein Structure Database** via their
REST API and render them in-notebook with **py3Dmol**, coloring each residue by its
conservation score (derived from per-column Shannon entropy of the MSA).

**Conservation score** (mapped from entropy):

$$
C_i = 1 - \frac{H_i}{\log_2 |\mathcal{A}|}
$$

where $H_i$ is the per-column entropy at MSA column $i$ and $|\mathcal{A}|=20$ is the
3Di alphabet size. $C_i=1$ means perfectly conserved; $C_i=0$ means maximum entropy.

**Color scheme**: a blue → white → red gradient where **red = highly conserved** and
**blue = variable**.

A **PyMOL `.pml` script** is also generated as a fallback for offline / publication-quality
rendering.

In [ ]:
# ---------- Conservation scores from per-column entropy ----------
# per_col_entropy is computed in Section 4 (shape: n_columns of the MSA)
# Normalise to conservation score: 1 = perfectly conserved, 0 = max entropy
max_entropy = np.log2(20)
conservation_scores = 1.0 - per_col_entropy / max_entropy
conservation_scores = np.clip(conservation_scores, 0, 1)

print(f"Conservation scores: {len(conservation_scores)} MSA columns")
print(f"  mean={conservation_scores.mean():.3f}  min={conservation_scores.min():.3f}  max={conservation_scores.max():.3f}")

In [ ]:
# ---------- Map MSA-column conservation to per-residue scores ----------
# For each sequence in the MSA subsample, map non-gap MSA columns to residue indices.
# We'll pick the first sequence as the demo structure target.

def msa_conservation_to_residue(msa_row, col_scores):
    """Map column-level conservation scores to residue-level for one MSA row.
    
    Returns list of (residue_index_0based, conservation_score) for non-gap positions.
    """
    residue_scores = []
    res_idx = 0
    for col_idx, ch in enumerate(msa_row):
        if ch != '-':
            score = col_scores[col_idx] if col_idx < len(col_scores) else 0.0
            residue_scores.append((res_idx, float(score)))
            res_idx += 1
    return residue_scores

# Use first sequence in the MSA subsample
demo_seq_idx = 0
demo_name = msa_ids[demo_seq_idx]
demo_msa_row = msa_seqs[demo_seq_idx]
residue_conservation = msa_conservation_to_residue(demo_msa_row, conservation_scores)

print(f"Demo sequence: {demo_name}")
print(f"  Residues with scores: {len(residue_conservation)}")
print(f"  First 10: {residue_conservation[:10]}")

In [ ]:
# ---------- Fetch AlphaFold structure + py3Dmol visualization ----------
import urllib.request, urllib.error, json as _json

def fetch_alphafold_pdb(uniprot_id):
    """Fetch a predicted PDB from AlphaFold DB. Returns PDB string or None."""
    url = f"https://alphafold.ebi.ac.uk/files/AF-{uniprot_id}-F1-model_v4.pdb"
    try:
        with urllib.request.urlopen(url, timeout=30) as resp:
            return resp.read().decode()
    except (urllib.error.URLError, urllib.error.HTTPError) as e:
        print(f"  Could not fetch AlphaFold structure for {uniprot_id}: {e}")
        return None

def extract_uniprot_id(seq_name):
    """Try to extract a UniProt accession from a FASTA header.
    
    Handles formats like:
      sp|P12345|NAME  ->  P12345
      tr|A0A0...|NAME ->  A0A0...
      P12345          ->  P12345
      >AF-P12345-F1   ->  P12345
    """
    import re
    # sp|XXXX|... or tr|XXXX|...
    m = re.search(r'[st][pr]\|([A-Za-z0-9]+)\|', seq_name)
    if m:
        return m.group(1)
    # AF-XXXX-F1
    m = re.search(r'AF-([A-Za-z0-9]+)-F', seq_name)
    if m:
        return m.group(1)
    # bare UniProt-like accession (6-10 alphanum)
    m = re.search(r'\b([OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z][0-9](?:[A-Z][A-Z0-9]{2}[0-9]){1,2})\b', seq_name)
    if m:
        return m.group(1)
    return None

# Try to resolve a UniProt ID from the demo sequence name
demo_uniprot = extract_uniprot_id(demo_name)
print(f"Sequence name: {demo_name}")
print(f"Extracted UniProt ID: {demo_uniprot}")

pdb_string = None
if demo_uniprot:
    print(f"Fetching AlphaFold structure for {demo_uniprot}...")
    pdb_string = fetch_alphafold_pdb(demo_uniprot)
    if pdb_string:
        print(f"  Success! PDB length: {len(pdb_string)} chars")
else:
    print("Could not extract UniProt ID from sequence name.")
    print("You can manually set demo_uniprot = 'P12345' and re-run this cell.")

In [ ]:
# ---------- py3Dmol interactive visualization ----------
# Color residues by conservation: red (conserved) -> white -> blue (variable)

def conservation_color(score):
    """Map conservation score [0,1] to hex color. 1=red, 0.5=white, 0=blue."""
    if score >= 0.5:
        # white -> red
        t = (score - 0.5) * 2  # 0..1
        r = 255
        g = int(255 * (1 - t))
        b = int(255 * (1 - t))
    else:
        # blue -> white
        t = score * 2  # 0..1
        r = int(255 * t)
        g = int(255 * t)
        b = 255
    return f"0x{r:02x}{g:02x}{b:02x}"

if pdb_string:
    try:
        import py3Dmol
        
        view = py3Dmol.view(width=800, height=600)
        view.addModel(pdb_string, "pdb")
        
        # Base style: cartoon, light grey
        view.setStyle({"cartoon": {"color": "lightgrey"}})
        
        # Color each residue by conservation
        for res_idx, score in residue_conservation:
            color = conservation_color(score)
            view.addStyle(
                {"resi": res_idx + 1},  # PDB residues are 1-indexed
                {"cartoon": {"color": color}}
            )
        
        # Highlight conserved blocks as thick tubes
        for block in conserved_blocks:
            start_col, end_col = block["start"], block["end"]
            # Map MSA columns to approximate residue range
            view.addStyle(
                {"resi": [start_col + 1, end_col + 1], "byres": True},
                {"cartoon": {"color": "red", "thickness": 0.4}}
            )
        
        view.zoomTo()
        view.show()
        print("py3Dmol visualization rendered above.")
        print("Red = highly conserved, Blue = variable, White = intermediate")
        
    except ImportError:
        print("py3Dmol not installed. Install with: pip install py3Dmol")
        print("Falling back to PyMOL script generation (next cell).")
else:
    print("No PDB structure available. Skipping py3Dmol visualization.")
    print("Set demo_uniprot manually and re-run the fetch cell, or use the PyMOL script below.")

In [ ]:
# ---------- PyMOL fallback script generation ----------
# Generates a .pml script that can be opened in PyMOL for publication-quality rendering

def generate_pymol_script(uniprot_id, residue_scores, conserved_blocks, output_path="visualize_conservation.pml"):
    """Generate a PyMOL .pml script that colors structure by conservation."""
    
    def score_to_rgb(score):
        """Same blue-white-red gradient as py3Dmol."""
        if score >= 0.5:
            t = (score - 0.5) * 2
            return (1.0, 1.0 - t, 1.0 - t)
        else:
            t = score * 2
            return (t, t, 1.0)
    
    lines = [
        "# PyMOL conservation visualization script",
        f"# Generated for AlphaFold structure: AF-{uniprot_id}-F1",
        "#",
        "# Usage: pymol visualize_conservation.pml",
        "",
        "# Fetch structure from AlphaFold DB",
        f'cmd.load("https://alphafold.ebi.ac.uk/files/AF-{uniprot_id}-F1-model_v4.pdb", "protein")',
        "",
        "# Basic display settings",
        "bg_color white",
        "hide everything",
        "show cartoon, protein",
        "set cartoon_smooth_loops, 1",
        "set cartoon_fancy_helices, 1",
        "set antialias, 2",
        "set ray_shadows, 1",
        "",
        "# Color all residues light grey as base",
        "color grey80, protein",
        "",
        "# --- Per-residue conservation coloring ---",
    ]
    
    # Group residues by similar scores to reduce commands
    score_bins = {}
    for res_idx, score in residue_scores:
        bin_key = round(score, 2)
        score_bins.setdefault(bin_key, []).append(res_idx + 1)  # 1-indexed
    
    for score_val, residues in sorted(score_bins.items()):
        r, g, b = score_to_rgb(score_val)
        color_name = f"cons_{int(score_val*100):03d}"
        resi_str = "+".join(str(r) for r in residues)
        lines.append(f'cmd.set_color("{color_name}", [{r:.3f}, {g:.3f}, {b:.3f}])')
        lines.append(f'color {color_name}, resi {resi_str}')
    
    # Highlight conserved blocks with thicker representation
    lines.append("")
    lines.append("# --- Conserved blocks (thicker cartoon) ---")
    for i, block in enumerate(conserved_blocks):
        start, end = block["start"] + 1, block["end"] + 1
        lines.append(f'create cons_block_{i}, protein and resi {start}-{end}')
        lines.append(f'show sticks, cons_block_{i}')
        lines.append(f'color red, cons_block_{i}')
        lines.append(f'set stick_radius, 0.15, cons_block_{i}')
    
    lines.extend([
        "",
        "# Final orientation and rendering",
        "orient protein",
        "zoom protein, 5",
        "",
        "# Uncomment to ray-trace a publication figure:",
        "# ray 2400, 1800",
        f'# png "conservation_{uniprot_id}.png", dpi=300',
        "",
        f'print("Conservation visualization loaded for AF-{uniprot_id}-F1")',
        f'print("Red = conserved, Blue = variable")',
    ])
    
    script = "\n".join(lines)
    with open(output_path, "w") as f:
        f.write(script)
    return output_path

if demo_uniprot:
    pml_path = generate_pymol_script(demo_uniprot, residue_conservation, conserved_blocks)
    print(f"PyMOL script written to: {pml_path}")
    print(f"Run with:  pymol {pml_path}")
    print()
    # Show first 30 lines as preview
    with open(pml_path) as f:
        preview = f.readlines()[:30]
    print("--- Script preview ---")
    for line in preview:
        print(line, end="")
else:
    print("No UniProt ID available. Set demo_uniprot and re-run.")

## 6. Dimensionality Reduction & Clustering of 3Di k-mer Profiles

Each 3Di sequence is represented as a **k-mer frequency vector** (k=3, yielding up to $20^3 = 8000$ dimensions).
We apply three complementary techniques to reveal natural groupings:

### 6a. PCA (Principal Component Analysis)
Linear projection onto the top principal components that maximise variance:

$$
\mathbf{Z} = \mathbf{X}_c \, \mathbf{V}_r, \quad \mathbf{X}_c = \mathbf{X} - \bar{\mathbf{x}}
$$

where $\mathbf{V}_r$ contains the top-$r$ right singular vectors.

### 6b. t-SNE / UMAP
Non-linear embeddings that preserve local neighbourhood structure:

- **t-SNE** minimises $\mathrm{KL}(P \| Q)$ where $P$ encodes high-dimensional affinities and $Q$ uses Student-$t$ kernels in 2D.
- **UMAP** constructs a fuzzy simplicial set in high-D and optimises a cross-entropy objective to find a low-D layout.

### 6c. Hierarchical Clustering Dendrogram
Ward's linkage on the k-mer vectors with the dendrogram truncated to show major clusters:

$$
\Delta(A,B) = \frac{n_A n_B}{n_A + n_B} \| \bar{\mathbf{x}}_A - \bar{\mathbf{x}}_B \|^2
$$

In [ ]:
# ---------- Build k-mer feature matrix for ALL sequences ----------
from collections import Counter
from itertools import product as iterproduct

K_DIM = 3  # k-mer size (same as Section 2b)
ALPHABET = sorted(set("ACDEFGHIKLMNPQRSTVWY"))

# Canonical list of all possible k-mers (fixed order for consistent vectors)
all_kmers_list = ["".join(c) for c in iterproduct(ALPHABET, repeat=K_DIM)]
kmer_to_idx = {km: i for i, km in enumerate(all_kmers_list)}
N_FEATURES = len(all_kmers_list)

def kmer_vector(seq, k=K_DIM):
    """Return normalised k-mer frequency vector of length 20^k."""
    counts = Counter(seq[i:i+k] for i in range(len(seq) - k + 1))
    total = sum(counts.values())
    vec = np.zeros(N_FEATURES)
    if total == 0:
        return vec
    for km, cnt in counts.items():
        if km in kmer_to_idx:
            vec[kmer_to_idx[km]] = cnt / total
    return vec

# Subsample for speed if dataset is large
MAX_DR = 2000
np.random.seed(42)
if len(sequences) > MAX_DR:
    dr_idx = np.sort(np.random.choice(len(sequences), MAX_DR, replace=False))
else:
    dr_idx = np.arange(len(sequences))

dr_names = [names[i] for i in dr_idx]
dr_seqs = [sequences[i] for i in dr_idx]

print(f"Building {K_DIM}-mer vectors for {len(dr_seqs)} sequences ({N_FEATURES} features)...")
X = np.array([kmer_vector(s) for s in dr_seqs])
print(f"Feature matrix shape: {X.shape}")
print(f"Non-zero features per sequence: mean={np.count_nonzero(X, axis=1).mean():.0f}")

In [ ]:
# ---------- 6a. PCA ----------
from sklearn.decomposition import PCA

pca = PCA(n_components=min(50, X.shape[0], X.shape[1]))
X_pca = pca.fit_transform(X)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scree plot
ax = axes[0]
cumvar = np.cumsum(pca.explained_variance_ratio_) * 100
ax.bar(range(1, len(pca.explained_variance_ratio_) + 1),
       pca.explained_variance_ratio_ * 100, color="steelblue", alpha=0.7, label="Individual")
ax.plot(range(1, len(cumvar) + 1), cumvar, "r-o", markersize=3, label="Cumulative")
ax.set_xlabel("Principal Component")
ax.set_ylabel("Explained Variance (%)")
ax.set_title("PCA Scree Plot")
ax.legend()
ax.set_xlim(0.5, 20.5)

# PC1 vs PC2 scatter
ax = axes[1]
# Color by sequence length as a proxy for structural complexity
lengths_dr = np.array([len(s) for s in dr_seqs])
sc = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=lengths_dr, cmap="viridis",
                s=8, alpha=0.6, edgecolors="none")
plt.colorbar(sc, ax=ax, label="Sequence length")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
ax.set_title("PCA — 3Di k-mer profiles")

plt.tight_layout()
plt.show()
print(f"Top 5 PCs explain {cumvar[4]:.1f}% of variance")

In [ ]:
# ---------- 6b. t-SNE and UMAP ----------
from sklearn.manifold import TSNE

# Use PCA-reduced data (top 50 PCs) as input for speed
X_pca50 = X_pca[:, :min(50, X_pca.shape[1])]

# t-SNE
print("Running t-SNE...")
tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
X_tsne = tsne.fit_transform(X_pca50)
print(f"  t-SNE done. KL divergence: {tsne.kl_divergence_:.4f}")

# UMAP (with fallback)
X_umap = None
try:
    import umap
    print("Running UMAP...")
    reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42)
    X_umap = reducer.fit_transform(X_pca50)
    print("  UMAP done.")
except ImportError:
    print("  umap-learn not installed (pip install umap-learn). Skipping UMAP.")

# Plot
n_plots = 2 if X_umap is not None else 1
fig, axes = plt.subplots(1, n_plots, figsize=(7 * n_plots, 6))
if n_plots == 1:
    axes = [axes]

# Hierarchical cluster labels for coloring (Ward, k=8 clusters)
from scipy.cluster.hierarchy import linkage, fcluster
Z_ward = linkage(X_pca50, method="ward")
N_CLUSTERS = 8
cluster_labels = fcluster(Z_ward, t=N_CLUSTERS, criterion="maxclust")

cmap = plt.cm.get_cmap("tab10", N_CLUSTERS)

ax = axes[0]
for cl in range(1, N_CLUSTERS + 1):
    mask = cluster_labels == cl
    ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1], s=8, alpha=0.6,
               label=f"Cluster {cl}", color=cmap(cl - 1), edgecolors="none")
ax.set_title("t-SNE of 3Di k-mer profiles")
ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")
ax.legend(markerscale=3, fontsize=7, loc="best")

if X_umap is not None:
    ax = axes[1]
    for cl in range(1, N_CLUSTERS + 1):
        mask = cluster_labels == cl
        ax.scatter(X_umap[mask, 0], X_umap[mask, 1], s=8, alpha=0.6,
                   label=f"Cluster {cl}", color=cmap(cl - 1), edgecolors="none")
    ax.set_title("UMAP of 3Di k-mer profiles")
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")
    ax.legend(markerscale=3, fontsize=7, loc="best")

plt.tight_layout()
plt.show()

print(f"\nCluster sizes (Ward, k={N_CLUSTERS}):")
for cl in range(1, N_CLUSTERS + 1):
    print(f"  Cluster {cl}: {(cluster_labels == cl).sum()} sequences")

In [ ]:
# ---------- 6c. Hierarchical Clustering Dendrogram ----------
from scipy.cluster.hierarchy import dendrogram

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Ward linkage dendrogram (truncated)
ax = axes[0]
dendrogram(Z_ward, truncate_mode="lastp", p=30, ax=ax,
           leaf_rotation=90, leaf_font_size=8,
           color_threshold=Z_ward[-(N_CLUSTERS - 1), 2])
ax.set_title(f"Ward Linkage Dendrogram (truncated to 30 leaves)")
ax.set_xlabel("Cluster (size)")
ax.set_ylabel("Ward distance")

# Average linkage for comparison
Z_avg = linkage(X_pca50, method="average")
ax = axes[1]
dendrogram(Z_avg, truncate_mode="lastp", p=30, ax=ax,
           leaf_rotation=90, leaf_font_size=8)
ax.set_title("Average Linkage Dendrogram (truncated to 30 leaves)")
ax.set_xlabel("Cluster (size)")
ax.set_ylabel("Average distance")

plt.tight_layout()
plt.show()

# Cluster characterisation summary
print(f"\n{'Cluster':>8} {'Size':>6} {'Mean len':>9} {'Std len':>8} {'Mean entropy':>13}")
print("-" * 50)
lengths_all = np.array([len(s) for s in dr_seqs])
entropies_all = np.array([
    -sum((c / len(s)) * np.log2(c / len(s)) for c in Counter(s).values())
    if len(s) > 0 else 0.0
    for s in dr_seqs
])
for cl in range(1, N_CLUSTERS + 1):
    mask = cluster_labels == cl
    print(f"{cl:>8} {mask.sum():>6} {lengths_all[mask].mean():>9.1f} "
          f"{lengths_all[mask].std():>8.1f} {entropies_all[mask].mean():>13.3f}")